# MS thesis project demo

This notebook demonstrates the speech degradation pipeline and compares the project's Whisper, dual-view fusion, and FastConformer ASR models on the **same randomly selected audio clips**.

Run it from the repository's Python environment. The degradation demo also needs `ffmpeg`; additive noise is used when the configured DEMAND noise index is present. Model-loading time is intentionally excluded from transcription timing.

In [ ]:
from __future__ import annotations

import gc
import html
import os
import random
import shutil
import sys
import time
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, Markdown, display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'ml').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the repository root from the current directory.')


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Repository: {REPO_ROOT}')
print(f'Inference device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))

## Configuration

Edit this cell before running the demos. Clip directories may be absolute or relative to the repository. Whisper sources may be local directories or Hugging Face model IDs; fusion and FastConformer checkpoints are local files.

In [ ]:
# Reproducible random sampling
RANDOM_SEED = 1337

# Part 1: one or more selected /clips directories
DEGRADATION_CLIPS_DIRS = [Path('data/cv-corpus-25.0/clips')]
DEGRADATION_CONFIG_PATH = Path('configs/speech_enhancement/degradation.yaml')
DEGRADATION_AUDIO_COUNT = 6
# Balanced as far as the count permits; valid values: mild, moderate, strong
DEGRADATION_SEVERITIES = ('mild', 'moderate', 'strong')

# Parts 2-4: sampled once, then reused by every ASR model
ASR_CLIPS_DIRS = [Path('data/cv-corpus-25.0/clips')]
ASR_AUDIO_COUNT = 5
ASR_BATCH_SIZE = 1
SAMPLE_RATE = 16_000

# Whisper
WHISPER_MODEL_PATH = 'models/asr/whisper-small/runs/whisper-small-fa/final'
WHISPER_PROCESSOR_PATH = 'openai/whisper-small'
WHISPER_LANGUAGE = 'Persian'
WHISPER_TASK = 'transcribe'
WHISPER_MAX_NEW_TOKENS = 225

# Dual-view fusion checkpoints are PyTorch state dictionaries, not standalone
# Hugging Face model directories. The loader first constructs a Whisper model
# from FUSION_BASE_ASR_PATH, then applies the fusion checkpoint weights. A Stage-2
# checkpoint normally includes and overwrites the Whisper weights, so the base
# source mainly provides architecture + generation configuration. A backbone-free
# Stage-1 checkpoint does not include those weights, so its base path must be the
# exact fine-tuned Whisper backbone used during fusion training. MODEL_NAME supplies
# the Whisper log-Mel feature extractor; PROCESSOR_PATH supplies the tokenizer.
FUSION_CHECKPOINT_PATH = Path(
    'artifacts/speech_enhancement/fusion/run_001/checkpoints/stage2_joint/fusion_model.pt'
)
FUSION_BASE_ASR_PATH = 'openai/whisper-small'
FUSION_MODEL_NAME = 'openai/whisper-small'
FUSION_PROCESSOR_PATH = 'openai/whisper-small'

# FastConformer: converted .pt bundle or original .nemo archive
FASTCONFORMER_MODEL_PATH = Path('models/stt_fa_fastconformer_ctc.pt')
FASTCONFORMER_MAX_BATCH_SECONDS = 30.0

In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.flac', '.ogg', '.opus', '.mp3', '.m4a'}


def repo_path(path: str | Path) -> Path:
    path = Path(path).expanduser()
    return path.resolve() if path.is_absolute() else (REPO_ROOT / path).resolve()


def existing_file(path: str | Path, label: str) -> Path:
    resolved = repo_path(path)
    if not resolved.is_file():
        raise FileNotFoundError(f'{label} not found: {resolved}')
    return resolved


def model_source(value: str | Path) -> str:
    # Resolve existing local paths while leaving Hugging Face IDs unchanged.
    expanded = Path(value).expanduser()
    for candidate in (expanded, REPO_ROOT / expanded):
        if candidate.exists():
            return str(candidate.resolve())
    text = str(value)
    if expanded.is_absolute() or text.startswith(('./', '../', 'models/', 'artifacts/')):
        raise FileNotFoundError(f'Local model source not found: {repo_path(expanded)}')
    return text


def audio_file_candidates(clips_dirs: Sequence[str | Path]) -> list[Path]:
    files: set[Path] = set()
    for raw_dir in clips_dirs:
        clips_dir = repo_path(raw_dir)
        if not clips_dir.is_dir():
            raise FileNotFoundError(f'Clips directory not found: {clips_dir}')
        for path in clips_dir.rglob('*'):
            if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
                files.add(path)
    return sorted(files)


def sample_audio_paths(clips_dirs: Sequence[str | Path], count: int, seed: int) -> list[Path]:
    if count < 1:
        raise ValueError('Audio count must be at least 1.')
    candidates = audio_file_candidates(clips_dirs)
    if len(candidates) < count:
        raise ValueError(f'Requested {count} clips, but found only {len(candidates)} audio-file candidates.')
    # Probe candidates in deterministic random order and stop as soon as enough
    # readable files are found. Common Voice can contain hundreds of thousands
    # of clips, so opening every file merely to choose a small sample is costly.
    random.Random(seed).shuffle(candidates)
    selected: list[Path] = []
    for path in candidates:
        try:
            sf.info(str(path))
        except (RuntimeError, sf.LibsndfileError):
            continue
        selected.append(path.resolve())
        if len(selected) == count:
            return selected
    raise ValueError(f'Requested {count} clips, but found only {len(selected)} readable files.')


def synchronize_device() -> None:
    if DEVICE == 'cuda':
        torch.cuda.synchronize()


def release_model(model: Any) -> None:
    model.to('cpu')
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def batched(items: Sequence[Any], batch_size: int) -> Iterable[Sequence[Any]]:
    if batch_size < 1:
        raise ValueError('ASR_BATCH_SIZE must be at least 1.')
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def show_transcriptions(
    title: str,
    paths: Sequence[Path],
    texts: Sequence[str],
    elapsed_seconds: float,
    details: Sequence[str | None] | None = None,
) -> None:
    if len(paths) != len(texts):
        raise ValueError('Each sampled audio must have one transcription.')
    print(f'{title}: {elapsed_seconds:.3f} s total; {elapsed_seconds / len(paths):.3f} s/audio')
    details = details or [None] * len(paths)
    for index, (path, text, detail) in enumerate(zip(paths, texts, details, strict=True), start=1):
        body = [f'### {index}. `{html.escape(path.name)}`', f'**Transcription:** {html.escape(text)}']
        if detail:
            body.append(detail)
        display(Markdown('\n\n'.join(body)))
        display(Audio(filename=str(path)))

## 1. Random degradation demo

Severity labels control parameter ranges, rather than depending on an accidental all-easy or all-hard random draw. The remaining pipeline choices stay random and reproducible. The original file is played directly; the degraded output is generated in memory and is not added to the repository.

In [ ]:
from ml.speech_data.generate_degraded_pairs import (
    ManifestItem,
    deep_merge,
    default_config,
    degrade_item,
    ffmpeg_encoder_listing,
    load_asset_index,
    load_config,
    resolve_ffmpeg_encoder,
    resolve_path,
)

SEVERITY_OVERRIDES: dict[str, dict[str, Any]] = {
    'mild': {
        'noise': {'probability': 0.45, 'second_scene_probability': 0.0, 'snr_buckets': [[15, 25]]},
        'level': {'gain_db': [-2, 2], 'clipping': {'enabled': False}},
        'network_impairment': {
            'enabled': True, 'mode': 'decoded_waveform_dropout', 'probability': 0.20,
            'loss_rate_buckets': [[0.003, 0.01]], 'burst_length': [1, 2],
        },
    },
    'moderate': {
        'noise': {'probability': 0.75, 'second_scene_probability': 0.10, 'snr_buckets': [[5, 15]]},
        'level': {'gain_db': [-4, 4], 'clipping': {'enabled': False}},
        'network_impairment': {
            'enabled': True, 'mode': 'decoded_waveform_dropout', 'probability': 0.55,
            'loss_rate_buckets': [[0.01, 0.05]], 'burst_length': [1, 4],
        },
    },
    'strong': {
        'noise': {'probability': 0.95, 'second_scene_probability': 0.25, 'snr_buckets': [[-5, 5]]},
        'level': {
            'gain_db': [-6, 6],
            'clipping': {'enabled': True, 'probability': 0.30, 'mode': 'hard', 'threshold': [0.75, 0.90]},
        },
        'network_impairment': {
            'enabled': True, 'mode': 'decoded_waveform_dropout', 'probability': 0.85,
            'loss_rate_buckets': [[0.05, 0.12]], 'burst_length': [2, 6],
        },
    },
}


def supported_codecs(entries: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:
    # A local ffmpeg build may omit AMR/GSM encoders. Keep every configured
    # codec it can actually encode, plus the pipeline's pass-through option.
    listing = ffmpeg_encoder_listing() if shutil.which('ffmpeg') else ''
    supported = [
        dict(entry) for entry in entries
        if entry['codec'] == 'pass_through' or resolve_ffmpeg_encoder(str(entry['codec']), listing) is not None
    ]
    if not supported:
        raise RuntimeError('No usable degradation codec was found; install ffmpeg and its audio encoders.')
    return supported


def severity_config(base_config: dict[str, Any], severity: str) -> dict[str, Any]:
    if severity not in SEVERITY_OVERRIDES:
        raise ValueError(f'Unknown severity {severity!r}; choose from {tuple(SEVERITY_OVERRIDES)}.')
    # Disable named profiles here because they intentionally override noise/loss
    # settings; the demo severity range should remain the controlling factor.
    return deep_merge(
        base_config,
        {'profiles': None, 'codec_distribution': supported_codecs(base_config['codec_distribution']),
         **SEVERITY_OVERRIDES[severity]},
    )


def degradation_details(metadata: dict[str, Any]) -> str:
    network = metadata['network_impairment']
    clipping = metadata['clipping']
    noise = 'none' if metadata['snr_db'] is None else f"{metadata['snr_db']:.1f} dB SNR ({', '.join(map(str, metadata['noise_scenes']))})"
    network_text = 'off' if not network['enabled'] else (
        f"{network['mode']}, target loss {100 * network['loss_rate']:.1f}%, "
        f"observed {100 * network['observed_loss_rate']:.1f}%"
    )
    clipping_text = 'off' if not clipping['enabled'] else (
        f"hard at {clipping['threshold']:.2f} ({100 * clipping['clipped_fraction']:.2f}% samples)"
    )
    bitrate = f" at {metadata['codec_bitrate']}" if metadata['codec_bitrate'] else ''
    return (
        f"- Severity range: **{metadata['severity']}**\n"
        f"- Additive noise: {noise}\n"
        f"- Gain: {metadata['gain_db']:+.1f} dB; clipping: {clipping_text}\n"
        f"- Channel: {metadata['channel_path']} ({metadata['channel_sample_rate']} Hz, "
        f"{metadata['channel_bandpass_hz'][0]:.0f}-{metadata['channel_bandpass_hz'][1]:.0f} Hz)\n"
        f"- Codec: {metadata['codec']}{bitrate}\n"
        f"- Network impairment: {network_text}"
    )

In [ ]:
config_path = existing_file(DEGRADATION_CONFIG_PATH, 'Degradation config')
base_config = default_config(load_config(config_path))
base_config['seed'] = RANDOM_SEED

noise_index = resolve_path(base_config.get('noise_index'), REPO_ROOT)
noise_assets = load_asset_index(noise_index) if noise_index and noise_index.is_file() else []
if not noise_assets:
    print('Note: no noise index was found; codec/channel/loss degradation still runs, but additive noise is skipped.')

selected_degradation_paths = sample_audio_paths(
    DEGRADATION_CLIPS_DIRS, DEGRADATION_AUDIO_COUNT, RANDOM_SEED
)
severity_order = [DEGRADATION_SEVERITIES[i % len(DEGRADATION_SEVERITIES)] for i in range(len(selected_degradation_paths))]
random.Random(RANDOM_SEED + 1).shuffle(severity_order)

for index, (audio_path, severity) in enumerate(zip(selected_degradation_paths, severity_order, strict=True), start=1):
    config = severity_config(base_config, severity)
    item = ManifestItem(id=f'{audio_path.stem}_{index}', split='demo', clean_path=audio_path)
    metadata, _clean_target, degraded, output_rate = degrade_item(
        item, variant_index=index - 1, config=config, noise_assets=noise_assets
    )
    metadata['severity'] = severity
    display(Markdown(f"### {index}. `{html.escape(audio_path.name)}`\n\n**Original audio**"))
    display(Audio(filename=str(audio_path)))
    display(Markdown('**Degraded audio**\n\n' + degradation_details(metadata)))
    display(Audio(data=degraded, rate=output_rate))

## Shared ASR sample

Run this cell once before the next three sections. They all consume `shared_asr_paths`, so the comparison cannot silently use different random clips. Re-run this cell with another seed when you want a new comparison set.

In [ ]:
shared_asr_paths = sample_audio_paths(ASR_CLIPS_DIRS, ASR_AUDIO_COUNT, RANDOM_SEED + 10_000)
print(f'Selected {len(shared_asr_paths)} shared ASR clips:')
for index, path in enumerate(shared_asr_paths, start=1):
    duration = sf.info(str(path)).duration
    display(Markdown(f'### {index}. `{html.escape(path.name)}` — {duration:.2f} s'))
    display(Audio(filename=str(path)))

## 2. Whisper transcription

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

from ml.asr.whisper_features import waveform_to_log_mel
from ml.fusion.train_fusion import configure_whisper_generation
from ml.utils.audio import load_audio, resample_audio

whisper_source = model_source(WHISPER_MODEL_PATH)
processor_source = model_source(WHISPER_PROCESSOR_PATH)
whisper_processor = WhisperProcessor.from_pretrained(
    processor_source, language=WHISPER_LANGUAGE, task=WHISPER_TASK
)
whisper_model = WhisperForConditionalGeneration.from_pretrained(whisper_source).to(DEVICE).eval()
configure_whisper_generation(whisper_model, WHISPER_LANGUAGE, WHISPER_TASK)

whisper_texts: list[str] = []
synchronize_device()
started = time.perf_counter()
for path_batch in batched(shared_asr_paths, ASR_BATCH_SIZE):
    features = []
    for path in path_batch:
        waveform, source_rate = load_audio(path)
        if source_rate != SAMPLE_RATE:
            waveform = resample_audio(waveform, source_rate, SAMPLE_RATE)
        features.append(waveform_to_log_mel(waveform, sample_rate=SAMPLE_RATE, model_name=processor_source))
    input_features = torch.stack(features).to(DEVICE)
    with torch.inference_mode(), torch.amp.autocast('cuda', enabled=DEVICE == 'cuda'):
        token_ids = whisper_model.generate(input_features=input_features, max_new_tokens=WHISPER_MAX_NEW_TOKENS)
    whisper_texts.extend(whisper_processor.batch_decode(token_ids, skip_special_tokens=True))
synchronize_device()
whisper_elapsed = time.perf_counter() - started

show_transcriptions('Whisper transcription time', shared_asr_paths, whisper_texts, whisper_elapsed)
release_model(whisper_model)
del whisper_model

## 3. Dual-view fusion transcription

The learned gate is averaged over time and encoder channels for each clip. `Enhanced/clean` is the weight `g`; `raw/noisy` is `1 - g`. These describe representation use and are not probabilities that the input itself is clean or noisy.

In [ ]:
from transformers import WhisperTokenizer

from ml.asr.train_whisper_small import WhisperExample
from ml.fusion.eval_fusion import configure_generation, load_fusion_model, transcribe_examples

fusion_checkpoint = existing_file(FUSION_CHECKPOINT_PATH, 'Fusion checkpoint')
fusion_base_source = model_source(FUSION_BASE_ASR_PATH)
fusion_processor_source = model_source(FUSION_PROCESSOR_PATH)
fusion_tokenizer = WhisperTokenizer.from_pretrained(fusion_processor_source)
fusion_model, backbone_included = load_fusion_model(
    fusion_checkpoint,
    base_asr_checkpoint=fusion_base_source,
    model_name=FUSION_MODEL_NAME,
)
configure_generation(fusion_model, WHISPER_LANGUAGE, WHISPER_TASK)
print('Fusion backbone weights:', 'included in checkpoint' if backbone_included else fusion_base_source)

fusion_examples = [WhisperExample(audio_path=path, transcript='') for path in shared_asr_paths]
synchronize_device()
started = time.perf_counter()
fusion_texts, enhanced_weights = transcribe_examples(
    fusion_model,
    fusion_examples,
    fusion_tokenizer,
    device=DEVICE,
    sample_rate=SAMPLE_RATE,
    model_name=FUSION_MODEL_NAME,
    batch_size=ASR_BATCH_SIZE,
    generation_max_length=WHISPER_MAX_NEW_TOKENS,
    amp_enabled=DEVICE == 'cuda',
)
synchronize_device()
fusion_elapsed = time.perf_counter() - started

if len(enhanced_weights) != len(shared_asr_paths):
    raise RuntimeError('The fusion gate hook did not return one weight per audio clip.')
fusion_details = [
    f'**Fusion gate:** enhanced/clean `{weight:.3f}` · raw/noisy `{1.0 - weight:.3f}`'
    for weight in enhanced_weights
]
show_transcriptions(
    'Fusion transcription time', shared_asr_paths, fusion_texts, fusion_elapsed, fusion_details
)
release_model(fusion_model)
del fusion_model

## 4. FastConformer transcription

In [ ]:
from ml.asr.eval_fastconformer import load_fastconformer

fastconformer_checkpoint = existing_file(FASTCONFORMER_MODEL_PATH, 'FastConformer checkpoint')
fastconformer_model = load_fastconformer(fastconformer_checkpoint, DEVICE).eval()

synchronize_device()
started = time.perf_counter()
fastconformer_texts = fastconformer_model.transcribe(
    [str(path) for path in shared_asr_paths],
    batch_size=ASR_BATCH_SIZE,
    device=DEVICE,
    target_sr=SAMPLE_RATE,
    progress=True,
    max_batch_seconds=FASTCONFORMER_MAX_BATCH_SECONDS,
)
synchronize_device()
fastconformer_elapsed = time.perf_counter() - started

show_transcriptions(
    'FastConformer transcription time',
    shared_asr_paths,
    fastconformer_texts,
    fastconformer_elapsed,
)
release_model(fastconformer_model)
del fastconformer_model